Your task is to create a bert-base-classifier of vacancy areas based on their titles.

Each vacancy can have more than one area so it's **Multi-label classification** not Multiclass classification




In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from nltk.tokenize import word_tokenize
from string import punctuation
from tqdm import tqdm

In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/sctars/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sctars/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sctars/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, RandomSampler, Dataset, SequentialSampler
import random
import transformers

/Users/sctars/Desktop/dlhw3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Try two or more different bert-like models(different berts, robertas, debertas, modernberts etc. or any other transformer based model) (**2 points max**)
 your notebook should contain the training process of all your models!

Note that the interfaces of different models may differ slightly.

In [4]:
MODEL_NAME =  'bert-base-uncased'
MAX_SEQ_LENGTH = 32
RESULT_MODEL_PATH = './model.pt'

In [5]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed = 12
seed_everything(seed)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
device

device(type='cpu')

In [8]:
punctuation = set('!"$%&\'()*,-/:;<=>?@[\\]^_`{|}~') # убрал #

In [9]:
def clean(text):
    return ' '.join([token.lower() for token in word_tokenize(text) if token not in punctuation])

In [10]:
df = pd.read_csv('./dataset_2020.csv')
df.shape

(78909, 2)

Each vacancy can have more than one area separated be space

Exapmle:

Malware Analyst for Imunify Security,analyst it_security

In [11]:
df_train, df_test = train_test_split(df, train_size=0.9, random_state=42)
df_train, df_valid = train_test_split(df_train, train_size=0.8, random_state=42)

# Finish TextClassificationDataset (**1 point max**)

In [12]:
class TextClassificationDataset(Dataset):
    def __init__(self, data, tokenizer, bianizer):
        self.data = data
        self.tokenizer = tokenizer
        self.sentences = [clean(sent) for sent in data.title.tolist()]
        self.target = [labels.split() for labels in data.area.tolist()]
        self.bianizer = bianizer
        self.target_one_hot = torch.tensor(self.bianizer.transform(self.target), dtype=torch.float)
        # ToDo add if you need somethong else



    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.sentences[idx],
            padding = 'max_length', 
            max_length=MAX_SEQ_LENGTH, 
            truncation=True)
        encoded = {
            key: torch.tensor(value)
            for key, value in encoded.items()
        }
        
        return encoded, self.target_one_hot[idx]

In [13]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
binarizer = MultiLabelBinarizer()
labels_train = [labels.split() for labels in df_train.area.tolist()]
binarizer.fit(labels_train)


/Users/sctars/Desktop/dlhw3/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


,"classes classes: array-like of shape (n_classes,), default=NoneIndicates an ordering for the class labels.All entries should be unique (cannot contain duplicate classes).",None
,"sparse_output sparse_output: bool, default=FalseSet to True if output binary array is desired in CSR sparse format.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)A copy of the `classes` parameter when provided.Otherwise it corresponds to the sorted set of classes foundwhen fitting.","ndarray[object](29,)","['admin','analyst','architector',...,'sysadmin','teamlead','tester']"


In [14]:
lengths = [len(tokenizer.encode(clean(text))) for text in df_train.title]
print('Min length: {:,} tokens'.format(min(lengths)))
print('Max length: {:,} tokens'.format(max(lengths)))
print('Median length: {:,} tokens'.format(np.median(lengths)))

Min length: 3 tokens
Max length: 30 tokens
Median length: 6.0 tokens


In [15]:
truncated = lengths.count(32)

In [16]:
sent = "Hello, how are You DataLoader?"
print('Tokenized: ', tokenizer.tokenize(sent))

Tokenized:  ['hello', ',', 'how', 'are', 'you', 'data', '##load', '##er', '?']


In [17]:
tokenizer.convert_ids_to_tokens([101,102, 0])

['[CLS]', '[SEP]', '[PAD]']

In [18]:
binarizer.classes_

array(['admin', 'analyst', 'architector', 'assistant', 'consultant',
       'coord', 'data_engineer', 'data_scientist', 'designer',
       'devel_metodolog', 'devops', 'director', 'doc_writer',
       'it_security', 'machine_learner', 'manager', 'networks', 'owner',
       'programmer', 'proj_manager', 'public_writer', 'recruter', 'sales',
       'sap', 'specialist', 'support', 'sysadmin', 'teamlead', 'tester'],
      dtype=object)

In [19]:
batch_size = 16

train_dataset = TextClassificationDataset(df_train, tokenizer, binarizer)
train_sampler = RandomSampler(train_dataset)
train_dataloader =  DataLoader(train_dataset, sampler=train_sampler, batch_size=batch_size)

valid_dataset = TextClassificationDataset(df_valid, tokenizer, binarizer)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)

test_dataset = TextClassificationDataset(df_test, tokenizer, binarizer)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)


In [20]:
batch = next(iter(train_dataloader))

print(type(batch))
print(len(batch))
print(type(batch[0]))
print(type(batch[1]))

<class 'list'>
2
<class 'dict'>
<class 'torch.Tensor'>


In [21]:
print(type(batch[0]["input_ids"]))
print(batch[1].shape)

<class 'torch.Tensor'>
torch.Size([16, 29])


In [22]:
class BertForMultilabel(nn.Module):
    def __init__(self,  num_labels: int):
        super(BertForMultilabel, self).__init__()

        self.bert = transformers.AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, num_labels)


    def train_bert(self, train_bert_flag=True):
        for param in self.bert.parameters():
            param.requires_grad = train_bert_flag



    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        pooled_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask)['pooler_output']
        output = self.drop(pooled_output)
        return self.out(output)

In [23]:
num_labels = len(binarizer.classes_)
model = BertForMultilabel(num_labels)
model.to(device)

/Users/sctars/Desktop/dlhw3/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BertForMultilabel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

# Train your classifier with freezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch


In [52]:
from tqdm.auto import tqdm

def train(model, iterator, optimizer, criterion, scheduler=None):
     model.train() # перевод в режим обучения
     losses = []

     for encoded, labels in tqdm(iterator, desc="Training"):
          input_ids = encoded["input_ids"].to(device)
          attention_mask = encoded["attention_mask"].to(device)
          labels = labels.to(device)
          
          optimizer.zero_grad()

          preds = model(input_ids=input_ids, attention_mask=attention_mask)
          
          loss = criterion(preds, labels)
          loss.backward()
          optimizer.step()
          if scheduler is not None:
               scheduler.step()
          
          losses.append(loss.item())

     return np.mean(losses)


In [ ]:
def validate(model, iterator, criterion):
    model.eval()
    val_losses = []
    
    with torch.no_grad():
        
        for encoded, labels in iterator:
            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)
            labels = labels.to(device)
            preds = model(input_ids=input_ids, attention_mask=attention_mask)
            val_loss = criterion(preds, labels)
            
            val_losses.append(val_loss.item())
            
        return np.mean(val_losses)


    # ToDo write val loop

In [30]:
def logits_to_labels(logits):
    preds = nn.Sigmoid()(logits.view(-1, num_labels))
    preds = preds.to('cpu').numpy()>0.5
    return preds.tolist()

In [31]:
model.train_bert(False)  # заморозка

In [35]:
num_epochs = 3
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

In [39]:
print(len(train_dataset))     # Количество вакансий
print(len(train_dataloader))  # Количество батчей

56814
3551


In [38]:
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}", flush=True)

    train_loss = train(model, train_dataloader, optimizer, criterion)
    val_loss = validate(model, valid_dataloader, criterion)

    print(
        f"Train loss = {train_loss:.4f}, "
        f"Val loss = {val_loss:.4f}",
        flush=True
    )

    scheduler.step()
    print(f"Learning rate = {scheduler.get_last_lr()[0]:.6f}", flush=True)


Epoch 1/3


Training:   0%|          | 0/3551 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Training: 100%|██████████| 3551/3551 [38:14<00:00,  1.55it/s]


Train loss = 0.0737, Val loss = 0.0562
Learning rate = 0.001000

Epoch 2/3


Training: 100%|██████████| 3551/3551 [35:13<00:00,  1.68it/s]


Train loss = 0.0695, Val loss = 0.0520
Learning rate = 0.000500

Epoch 3/3


Training: 100%|██████████| 3551/3551 [35:55<00:00,  1.65it/s]


Train loss = 0.0669, Val loss = 0.0496
Learning rate = 0.000500


In [ ]:
torch.save(model.state_dict(), "./bert_frozen.pt")
RESULT_MODEL_PATH = "./bert_frozen.pt"


In [44]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device), weights_only=True))
test_preds = validate(model, test_dataloader, criterion)

возникли проблемы с test preds- переписала функцию

In [46]:
model.eval()
test_preds = []

with torch.no_grad():
    for encoded, labels in test_dataloader:
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        test_preds.append(preds.cpu().numpy())

test_preds = np.concatenate(test_preds, axis=0)

In [47]:
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

                 precision    recall  f1-score   support

          admin       0.00      0.00      0.00        61
        analyst       1.00      0.40      0.58       302
    architector       0.00      0.00      0.00       111
      assistant       0.00      0.00      0.00        14
     consultant       0.00      0.00      0.00        23
          coord       0.00      0.00      0.00        11
  data_engineer       0.00      0.00      0.00       136
 data_scientist       0.00      0.00      0.00       154
       designer       0.99      0.42      0.59       409
devel_metodolog       0.00      0.00      0.00        44
         devops       1.00      0.53      0.69       338
       director       0.00      0.00      0.00        17
     doc_writer       0.00      0.00      0.00        18
    it_security       0.00      0.00      0.00        54
machine_learner       0.00      0.00      0.00        42
        manager       1.00      0.01      0.01       427
       networks       0.00    

/Users/sctars/Desktop/dlhw3/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sctars/Desktop/dlhw3/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# Train your classifier with unfreezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch

In [48]:
epochs = 3
lr = 2e-5
WARMUP_PROPORTION = 0.1
warmup_steps = int(len(train_dataloader) * epochs * WARMUP_PROPORTION)

In [ ]:
model.train_bert(True)

In [50]:
t_total = len(train_dataloader) * epochs
no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
param_optimizer = list(model.named_parameters())
optimizer_grouped_parameters = [
        {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.001},
        {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=lr)
scheduler = transformers.get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=t_total
)

In [ ]:
best_val_loss = float("inf")

for epoch in range(epochs):
    train_loss = train(model, train_dataloader, optimizer, criterion, scheduler)
    val_loss = validate(model, valid_dataloader, criterion)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "./bert_unfrozen.pt")
    print(f"Epoch {epoch + 1}: train loss = {train_loss:.4f}, val loss = {val_loss:.4f}")


Training: 100%|██████████| 3551/3551 [2:02:43<00:00,  2.07s/it]  


Epoch 1: train loss = 0.0090, val loss = 0.0004


Training: 100%|██████████| 3551/3551 [2:02:34<00:00,  2.07s/it]  


Epoch 2: train loss = 0.0007, val loss = 0.0007


Training:  57%|█████▋    | 2023/3551 [1:06:34<1:11:56,  2.83s/it]

In [ ]:
RESULT_MODEL_PATH = "./bert_unfrozen.pt"

In [ ]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device), weights_only=True))
test_preds = validate(model, test_dataloader, criterion)

In [ ]:
model.eval()
test_preds = []

with torch.no_grad():
    for encoded, labels in test_dataloader:
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = (torch.sigmoid(logits) >= 0.5).int()
        test_preds.append(preds.cpu().numpy())

test_preds = np.concatenate(test_preds, axis=0)

In [ ]:
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

# Роберта!

In [ ]:
class RobertaForMultilabel(nn.Module):
    def __init__(self, num_labels):
        super().__init__()

        self.roberta = transformers.AutoModel.from_pretrained(
            "roberta-base"
        )

        self.drop = nn.Dropout(p=0.3)

        self.out = nn.Linear(
            self.roberta.config.hidden_size,
            num_labels
        )

    def train_roberta(self, flag=True):
        for param in self.roberta.parameters():
            param.requires_grad = flag
        
        
    def forward(self, input_ids=None, attention_mask=None):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        output = self.drop(cls_output)

        return self.out(output)

In [ ]:
MODEL_NAME = "roberta-base"

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
train_dataset = TextClassificationDataset(df_train, tokenizer, binarizer)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = TextClassificationDataset(df_valid, tokenizer, binarizer)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)

test_dataset = TextClassificationDataset(df_test, tokenizer, binarizer)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
model = RobertaForMultilabel(num_labels).to(device)

In [ ]:
model.train_roberta(False)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

In [ ]:
best_val_loss = float("inf")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}", flush=True)

    train_loss = train(model, train_dataloader, optimizer, criterion)
    val_loss = validate(model, valid_dataloader, criterion)

    if val_loss < best_val_loss:
        roberta_best_val_loss = val_loss
        torch.save(model.state_dict(), "./roberta_frozen.pt")
    
    print(
        f"Train loss = {train_loss:.4f}, "
        f"Val loss = {val_loss:.4f}",
        flush=True
    )

    scheduler.step()
    print(f"Learning rate = {scheduler.get_last_lr()[0]:.6f}", flush=True)

# Размораживаем

In [ ]:
model.train_roberta(True)

In [ ]:
lr = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [ ]:
epochs = 3

total_steps = len(train_dataloader) * epochs

scheduler = transformers.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

In [ ]:
best_val_loss = float("inf")

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}", flush=True)

    train_loss = train(
        model, train_dataloader, optimizer, criterion, scheduler
    )

    val_loss = validate(model, valid_dataloader, criterion)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "./roberta_unfrozen.pt")
        
    print(
        f"Train loss = {train_loss:.4f}, "
        f"Val loss = {val_loss:.4f}",
        flush=True
    )

In [ ]:
Модель Frozen Bert научилась хорошо распознавать несколько популярных категорий, 
но почти полностью игнорирует большинство остальных.
Метрика Macro F1 показала очень низкое качество по совокупности категорий (0,15), 
на weighted F1 результат лучше из-за распространенных классов (0,62)
Macro recall 0.12 - модель пропустила большинство положительных примеров во многих категориях.
Лучше всего распознались programmer, devops, designer, analyst и project manager

# Results (3 points max)

Write your conclusion

What models and what training parameters did you use?

What was the reason for your choice?

What were the results?

What metrics do you consider the most important?

In [ ]:
no_decay param choice reasoning: https://github.com/huggingface/transformers/issues/492